## Imports

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import math
import os
import time
import requests
import pandas as pd
import json
import numpy as np
from pathlib import Path
from typing import Optional, Dict, List, Union
from unidecode import unidecode
import matplotlib.pyplot as plt

import pandas_gbq
from google.auth import default
from google.cloud import bigquery
from google.api_core.exceptions import NotFound

In [3]:
from funcoes_escoragem import *

## Diretório

In [4]:
BASE_DIR = Path("data")
RAW_DIR = BASE_DIR / "raw"
TRUSTED_DIR = BASE_DIR / "trusted"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_DIR, TRUSTED_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# Funil Modelos

In [5]:
project_id = 'loft-dl-fintech'

In [36]:
query = '''
WITH tb_leads AS (
  SELECT 
    rf.contract_id,
    date(rf.dt_lead) as dt_lead,
    date(cf.requested_at) as requested_at,
    date(rf.dt_proposta_iniciada) as iniciada_at,
    date(rf.dt_proposta_enviada) as enviada_at,
    date(rf.activated_at) as activated_at,
    date(rf.cancelled_at) as cancelled_at,
    date(rf.dt_saida) as dt_saida,
    cf.tipo_contrato,
    rd.product_nm,
    cf.modeloBlend,
    cf.bureau_nm,
    cf.modelo_blend,
    case when cf.modeloBlend in ('BLEND_4', 'BVS_CUSTOM_V2', 'HFT1') then 'BLEND4'
         when cf.bureau_nm in ('BLEND_4', 'BVS_CUSTOM_V2', 'HFT1') then 'BLEND4'
         when cf.modeloBlend in ('BLEND3_3', 'BVS_CUSTOM', 'HVA4') then 'BLEND3'
         when cf.bureau_nm in ('BLEND3_3', 'BVS_CUSTOM', 'HVA4') then 'BLEND3'
         else modeloBlend end as bureau_nm_ajust,
    case when cf.modeloBlend in ('BVS_CUSTOM_V2', 'HFT1', 'BVS_CUSTOM', 'HVA4') then 1
         when cf.bureau_nm in ('BVS_CUSTOM_V2', 'HFT1', 'BVS_CUSTOM', 'HVA4') then 1 else 0 end as is_fallback,
    cast(rf.total_rental_value_informed_nr as FLOAT64) as total_rental_value_informed_nr,
    cast(rf.rental_value_nr as FLOAT64) as rental_value_nr,
    -- cf.agency_id,
    -- cf.contract_city_nm,
    -- cf.person_age,
    cf.qtd_proponentes,
    cf.score_imobiliaria,
    cf.person_restriction_total_value,
    ca.bvs_cust_score_nr,
    ca.blend_regressao_predict_nr,
    cf.rating_score_ds,
    rd.pre_analysis_result,
    rd.lead_elegivel,
    rd.proposta_iniciada,
    rd.proposta_enviada,
    rd.proposta_aprovada,
    rd.proposta_ativada,
    rd.is_activeted
  FROM loft-dl-fintech.cp_gold.requests_fact AS rf
  LEFT JOIN loft-dl-fintech.cp_gold.requests_dim AS rd
    ON rf.contract_id = rd.contract_id
  LEFT JOIN loft-dl-fintech.cp_gold.credit_fact AS cf
    ON rf.contract_id = cf.contract_id
  LEFT JOIN loft-dl-fintech.cp_silver.int_credit_analyses AS ca
    ON rf.contract_id = ca.contract_id
  WHERE cf.tipo_contrato = 'PF'
),
first_defaults AS (
  SELECT
    contract_id
    , DATE(MIN(pendency_created_at)) AS first_comunicacao_date
    , DATE(MIN(pendency_at)) AS first_competencia_date
  FROM loft-dl-fintech.cp_gold.watchlist_fact
  WHERE pendency_type = "Inadimplência"
  GROUP BY contract_id
)
SELECT
  l.*
  , fd.first_comunicacao_date
  , fd.first_competencia_date
FROM tb_leads AS l
LEFT JOIN first_defaults as fd
  on l.contract_id = fd.contract_id
  and l.activated_at is not null
where 1=1
    and date(l.requested_at) >= date('2026-07-01')
    and date(l.requested_at) < date(current_date())
'''

In [37]:
df_funil = pandas_gbq.read_gbq(query, project_id=project_id)
df_funil

Downloading: 100%|██████████|


,contract_id,dt_lead,requested_at,iniciada_at,enviada_at,activated_at,cancelled_at,dt_saida,tipo_contrato,product_nm,...,rating_score_ds,pre_analysis_result,lead_elegivel,proposta_iniciada,proposta_enviada,proposta_aprovada,proposta_ativada,is_activeted,first_comunicacao_date,first_competencia_date
0,4333052,2026-07-04,2026-07-04,NaT,NaT,NaT,2026-07-04,NaT,PF,None,...,E,REPROVAR,False,False,False,False,False,False,NaT,NaT
1,4333069,2026-07-04,2026-07-04,NaT,NaT,NaT,2026-07-04,NaT,PF,None,...,E,REPROVAR,False,False,False,False,False,False,NaT,NaT
2,4332800,2026-07-04,2026-07-04,NaT,NaT,NaT,2026-07-04,NaT,PF,None,...,E,REPROVAR,False,False,False,False,False,False,NaT,NaT
3,4332628,2026-07-04,2026-07-04,NaT,NaT,NaT,2026-07-04,NaT,PF,None,...,E,REPROVAR,False,False,False,False,False,False,NaT,NaT
4,4332643,2026-07-04,2026-07-04,NaT,NaT,NaT,NaT,NaT,PF,None,...,B,APROVAR,True,False,False,False,False,False,NaT,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23212,4335096,2026-07-06,2026-07-06,2026-07-06,2026-07-06,2026-07-06,NaT,NaT,PF,Smart Plus,...,B,APROVAR,True,True,True,True,True,True,NaT,NaT
23213,4334464,2026-07-06,2026-07-06,2026-07-06,2026-07-06,NaT,NaT,NaT,PF,Smart Plus,...,B,APROVAR,True,True,True,True,False,False,NaT,NaT
23214,4336729,2026-07-06,2026-07-06,2026-07-06,NaT,NaT,NaT,NaT,PF,Smart Plus,...,B,APROVAR,True,True,False,False,False,False,NaT,NaT
23215,4336443,2026-07-06,2026-07-06,2026-07-06,2026-07-06,NaT,NaT,NaT,PF,Smart Plus,...,A,DERIVAR,True,True,True,False,False,False,NaT,NaT


In [38]:
df_funil.groupby("bureau_nm_ajust", dropna=False).size()

bureau_nm_ajust
BLEND3                  20868
BLEND4                   1033
BLEND_REGRESSAO_2026     1305
NaN                        11
dtype: int64

In [39]:
df_funil.groupby("bureau_nm", dropna=False).size()

bureau_nm
                            3
BLEND3_3                19581
BLEND_4                   960
BLEND_REGRESSAO_2026     1211
BVS_CUSTOM                258
BVS_CUSTOM_V2              29
HFT1                        4
HVA3                      892
HVA4                      268
NaN                        11
dtype: int64

In [40]:
df_funil.groupby("bureau_nm_ajust", dropna=False).size()

bureau_nm_ajust
BLEND3                  20868
BLEND4                   1033
BLEND_REGRESSAO_2026     1305
NaN                        11
dtype: int64

In [41]:
df_funil.groupby("modeloBlend", dropna=False).size()

modeloBlend
BLEND3_3                20660
BLEND_4                  1000
BLEND_REGRESSAO_2026     1513
BVS_CUSTOM_V2              29
HFT1                        4
NaN                        11
dtype: int64

In [42]:
bvs = pd.to_numeric(df_funil["bvs_cust_score_nr"], errors="coerce")
score = pd.to_numeric(df_funil["blend_regressao_predict_nr"], errors="coerce")

conditions = [
    bvs <= 334,                         # corte customizado BVS → E
    score.between(763, 1000),           # 763 – 1000
    score.between(704, 762),            # 704 – 762
    score.between(653, 703),            # 653 – 703
    score.between(607, 652),            # 607 – 652
    score.between(562, 606),            # 562 – 606
    score.between(520, 561),            # 520 – 561
    score.between(480, 519),            # 480 – 519
    score.between(443, 479),            # 443 – 479
    score.between(408, 442),            # 408 – 442
    score.between(375, 407),            # 375 – 407
    score.between(343, 374),            # 343 – 374
    score.between(307, 342),            # 307 – 342
    score.between(0, 306),              # 0 – 306
]

choices = [
    "E.BVS",      # override BVS ≤ 334
    "1.A+",
    "2.A",
    "3.A",
    "4.B+",
    "5.B+",
    "6.B",
    "7.B",
    "8.C",
    "9.D+",
    "10.D",
    "11.D",
    "E-1.E",
    "E-2.E",
]

df_funil["rating_cl_pol_blend4"] = np.select(conditions, choices, default=None)

In [43]:
bvs = pd.to_numeric(df_funil["bvs_cust_score_nr"], errors="coerce")
score = pd.to_numeric(df_funil["blend_regressao_predict_nr"], errors="coerce")

conditions = [
    bvs <= 334,                         # corte customizado BVS → E
    score.between(763, 1000),           # 763 – 1000
    score.between(704, 762),            # 704 – 762
    score.between(653, 703),            # 653 – 703
    score.between(607, 652),            # 607 – 652
    score.between(562, 606),            # 562 – 606
    score.between(520, 561),            # 520 – 561
    score.between(480, 519),            # 480 – 519
    score.between(443, 479),            # 443 – 479
    score.between(408, 442),            # 408 – 442
    score.between(375, 407),            # 375 – 407
    score.between(343, 374),            # 343 – 374
    score.between(307, 342),            # 307 – 342
    score.between(0, 306),              # 0 – 306
]

choices = [
    "9.E.BVS",      # override BVS ≤ 334
    "1.A+",
    "2.A",
    "2.A",
    "3.B+",
    "3.B+",
    "4.B",
    "4.B",
    "5.C",
    "6.D+",
    "7.D",
    "7.D",
    "8.E",
    "8.E",
]

df_funil["rating_pol_blend4"] = np.select(conditions, choices, default=None)

In [44]:
df_funil.groupby("rating_pol_blend4", dropna=False).size()

rating_pol_blend4
1.A+        449
2.A        1915
3.B+       2398
4.B        2399
5.C        1228
6.D+       1218
7.D        2335
8.E        6411
9.E.BVS     906
NaN        3958
dtype: int64

In [45]:
df_funil.to_csv(ANALYTICS_DIR/"df_funil_blend4.csv", index=False)